In [ ]:
import ast
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

try:
    import wandb
except ImportError:
    wandb = None

BLOCK_SIZE = 64
SCRIPT_DIR = Path(__file__).resolve().parent
REPO_ROOT = SCRIPT_DIR.parent
CLASSIFIER_DIR = REPO_ROOT / "video-classifier"
ANNOTATIONS_PATH = REPO_ROOT / "data" / "annotations" / "annotations_train_test.csv"
EMBEDDINGS_DIR = [
    REPO_ROOT / "embeddings",
]
TEST_EMBEDDINGS_DIR = REPO_ROOT / "embeddings"
ARTIFACTS_DIR = CLASSIFIER_DIR / "models"
PREDICTIONS_PATH = ARTIFACTS_DIR / "test_predictions.txt"
WANDB_PROJECT = "action-classifier"
USE_WANDB = wandb is not None

VERB_CONFIG = {
    "block_size": BLOCK_SIZE,
    "epochs": 30,
    "batch_size": 256,
    "lr": 2e-4,
    "weight_decay": 3e-4,
    "dropout": 0.25,
    "hidden_dims": [1536,768],
    "test_size": 0.2,
    "random_state": 42,
}

NOUN_CONFIG = {
    "block_size": BLOCK_SIZE,
    "epochs": 30,
    "batch_size": 128,
    "lr": 2e-4,
    "weight_decay": 3e-4,
    "dropout": 0.4,
    "hidden_dims": [1536,768],
    "test_size": 0.2,
    "random_state": 42,
}

In [ ]:
# Load only relevant annotations
df = pd.read_csv(ANNOTATIONS_PATH)
df = df[df["relevant"] == True].copy()

# Convert frame ranges to block ranges
df["start_block"] = df["start_frame"] // BLOCK_SIZE
df["stop_block"] = df["stop_frame"] // BLOCK_SIZE

verb_class_to_name = df.drop_duplicates("verb_class").set_index("verb_class")["verb"].to_dict()
noun_class_to_name = {}
for _, row in df.iterrows():
    primary_noun_class = ast.literal_eval(row["all_noun_classes"])[0]
    noun_class_to_name.setdefault(primary_noun_class, row["noun"])

# Expand each action to one row per block it spans
rows = []
for _, row in df.iterrows():
    # all_noun_classes is a list like [3] or [1, 2, 5] - an action can have multiple nouns.
    # We use the first one as the primary noun class, which corresponds to the primary noun.
    # If a block has multiple overlapping actions, we later resolve the conflict by majority vote.
    primary_noun_class = ast.literal_eval(row["all_noun_classes"])[0]
    for block in range(row["start_block"], row["stop_block"] + 1):
        rows.append({
            "video_id": row["video_id"],
            "block": block,
            "verb_class": row["verb_class"],
            "noun_class": primary_noun_class,
        })

expanded = pd.DataFrame(rows)

# If multiple actions overlap the same block, pick the most frequent label
verb_labels = expanded.groupby(["video_id", "block"])["verb_class"].agg(lambda x: x.mode()[0])
noun_labels = expanded.groupby(["video_id", "block"])["noun_class"].agg(lambda x: x.mode()[0])

print(f"Relevant blocks: {len(verb_labels)}")
print(f"Unique verb classes: {verb_labels.nunique()}  Unique noun classes: {noun_labels.nunique()}")

In [ ]:
# Load embeddings only for blocks that have a label
all_embeddings = []
all_verb_labels = []
all_noun_labels = []
all_block_keys = []

for embeddings_dir in EMBEDDINGS_DIR:
    for pkl_path in sorted(embeddings_dir.glob("*.pkl")):
        video_id = pkl_path.stem

        with pkl_path.open("rb") as f:
            payload = pickle.load(f)

        embeddings = payload["embeddings"] if isinstance(payload, dict) else payload

        for block_idx in range(len(embeddings)):
            key = (video_id, block_idx)
            if key not in verb_labels.index:
                continue  # skip irrelevant blocks
            all_embeddings.append(embeddings[block_idx])
            all_verb_labels.append(verb_labels[key])
            all_noun_labels.append(noun_labels[key])
            all_block_keys.append(key)

X = torch.tensor(np.stack(all_embeddings), dtype=torch.float32)
block_keys = np.array(all_block_keys, dtype=object)

# Encode labels as contiguous integers starting from 0
verb_encoder = LabelEncoder().fit(all_verb_labels)
noun_encoder = LabelEncoder().fit(all_noun_labels)

y_verb = torch.tensor(verb_encoder.transform(all_verb_labels), dtype=torch.long)
y_noun = torch.tensor(noun_encoder.transform(all_noun_labels), dtype=torch.long)

n_verb_classes = len(verb_encoder.classes_)
n_noun_classes = len(noun_encoder.classes_)

print(f"Loaded {len(X)} relevant blocks")
print(f"Verb classes: {n_verb_classes}  Noun classes: {n_noun_classes}")

In [ ]:
def make_classifier(input_dim, num_classes, hidden_dims=None, dropout=0.25):
    hidden_dims = hidden_dims or [512, 256, 128]
    layers = []
    prev_dim = input_dim
    for hidden_dim in hidden_dims:
        layers.extend([
            nn.Linear(prev_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        ])
        prev_dim = hidden_dim
    layers.append(nn.Linear(prev_dim, num_classes))
    return nn.Sequential(*layers)

def train_model(model, X_train, y_train, X_val, y_val, device, task_name, config):
    model = model.to(device)
    loader = DataLoader(TensorDataset(X_train, y_train), batch_size=config["batch_size"], shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"])
    criterion = nn.CrossEntropyLoss()

    if USE_WANDB and wandb.run is not None:
        wandb.watch(model, criterion=criterion, log="gradients", log_freq=100)

    best_val_acc = 0.0
    for epoch in range(1, config["epochs"] + 1):
        model.train()
        total_loss = 0.0
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y_batch)

        train_loss = total_loss / len(y_train)
        val_loss, val_acc, _ = evaluate_model(model, X_val, y_val, device, criterion)
        best_val_acc = max(best_val_acc, val_acc)
        print(f"epoch={epoch:02d}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_acc={val_acc:.3f}")

        if USE_WANDB and wandb.run is not None:
            wandb.log({
                "epoch": epoch,
                f"{task_name}/train_loss": train_loss,
                f"{task_name}/val_loss": val_loss,
                f"{task_name}/val_acc": val_acc,
                f"{task_name}/best_val_acc": best_val_acc,
            })
    return model

def evaluate_model(model, X_eval, y_eval, device, criterion=None):
    model.eval()
    with torch.no_grad():
        logits = model(X_eval.to(device)).cpu()
        preds = logits.argmax(dim=-1)
        loss = criterion(logits, y_eval).item() if criterion is not None else None
    acc = (preds == y_eval).float().mean().item()
    return loss, acc, preds


def save_classifier_checkpoint(path, model, encoder, config, task_name):
    ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
    checkpoint = {
        "task_name": task_name,
        "model_state_dict": model.cpu().state_dict(),
        "input_dim": input_dim,
        "num_classes": len(encoder.classes_),
        "classes": encoder.classes_.tolist(),
        "hidden_dims": config["hidden_dims"],
        "dropout": config["dropout"],
        "config": dict(config),
        "block_size": BLOCK_SIZE,
    }
    torch.save(checkpoint, path)
    model.to(device)
    print(f"Saved {task_name} model to {path}")


def load_classifier_checkpoint(path, device):
    checkpoint = torch.load(path, map_location=device)
    model = make_classifier(
        checkpoint["input_dim"],
        checkpoint["num_classes"],
        checkpoint["hidden_dims"],
        checkpoint["dropout"],
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device).eval()

    encoder = LabelEncoder()
    encoder.classes_ = np.array(checkpoint["classes"])
    return model, encoder, checkpoint


def start_wandb_run(task_name, num_classes, num_samples, config_type):
    if not USE_WANDB:
        print("W&B disabled: install with `pip install wandb` to log runs.")
        return None
    run_config = config_type | {
        "task": task_name,
        "num_classes": num_classes,
        "num_samples": num_samples,
        "input_dim": input_dim,
        "device": str(device),
    }
    return wandb.init(project=WANDB_PROJECT, name=f"{task_name}-classifier", config=run_config, reinit=True)

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
input_dim = X.shape[1]
print(f"Device: {device}  Input dim: {input_dim}")

In [ ]:
# --- Verb classifier ---
# Drop classes with fewer than 2 samples - stratified split requires at least 2 per class
verb_counts = y_verb.bincount()
valid_verb_mask = verb_counts[y_verb] >= 2
X_verb, y_verb_filtered = X[valid_verb_mask], y_verb[valid_verb_mask]
print(f"Dropped {(~valid_verb_mask).sum().item()} blocks with singleton verb classes")

X_train, X_test, y_train, y_test = train_test_split(
    X_verb,
    y_verb_filtered,
    test_size=VERB_CONFIG["test_size"],
    random_state=VERB_CONFIG["random_state"],
    stratify=y_verb_filtered,
)

print("Training verb classifier...")
verb_run = start_wandb_run("verb", n_verb_classes, len(X_verb), VERB_CONFIG)
verb_model = train_model(
    make_classifier(input_dim, n_verb_classes, VERB_CONFIG["hidden_dims"], VERB_CONFIG["dropout"]),
    X_train,
    y_train,
    X_test,
    y_test,
    device,
    task_name="verb",
    config=VERB_CONFIG,
)

print("\nVerb classifier results:")
verb_loss, verb_acc, _ = evaluate_model(verb_model, X_test, y_test, device, nn.CrossEntropyLoss())
print(f"accuracy: {verb_acc:.3f}  ({int(verb_acc * len(y_test))}/{len(y_test)} correct)")
verb_model_path = ARTIFACTS_DIR / "verb_classifier.pt"
save_classifier_checkpoint(verb_model_path, verb_model, verb_encoder, VERB_CONFIG, "verb")
if USE_WANDB and wandb.run is not None:
    wandb.summary["final_loss"] = verb_loss
    wandb.summary["final_acc"] = verb_acc
    wandb.finish()

In [ ]:
# --- Noun classifier ---
# Drop classes with fewer than 2 samples - stratified split requires at least 2 per class
noun_counts = y_noun.bincount()
valid_noun_mask = noun_counts[y_noun] >= 2
X_noun, y_noun_filtered = X[valid_noun_mask], y_noun[valid_noun_mask]
print(f"Dropped {(~valid_noun_mask).sum().item()} blocks with singleton noun classes")

X_train, X_test, y_train, y_test = train_test_split(
    X_noun,
    y_noun_filtered,
    test_size=NOUN_CONFIG["test_size"],
    random_state=NOUN_CONFIG["random_state"],
    stratify=y_noun_filtered,
)

print("Training noun classifier...")
noun_run = start_wandb_run("noun", n_noun_classes, len(X_noun), NOUN_CONFIG)
noun_model = train_model(
    make_classifier(input_dim, n_noun_classes, NOUN_CONFIG["hidden_dims"], NOUN_CONFIG["dropout"]),
    X_train,
    y_train,
    X_test,
    y_test,
    device,
    task_name="noun",
    config=NOUN_CONFIG,
)

print("\nNoun classifier results:")
noun_loss, noun_acc, _ = evaluate_model(noun_model, X_test, y_test, device, nn.CrossEntropyLoss())
print(f"accuracy: {noun_acc:.3f}  ({int(noun_acc * len(y_test))}/{len(y_test)} correct)")
noun_model_path = ARTIFACTS_DIR / "noun_classifier.pt"
save_classifier_checkpoint(noun_model_path, noun_model, noun_encoder, NOUN_CONFIG, "noun")
if USE_WANDB and wandb.run is not None:
    wandb.summary["final_loss"] = noun_loss
    wandb.summary["final_acc"] = noun_acc
    wandb.finish()

In [ ]:
# Load saved models and export predictions for the test embeddings
verb_model, loaded_verb_encoder, _ = load_classifier_checkpoint(ARTIFACTS_DIR / "verb_classifier.pt", device)
noun_model, loaded_noun_encoder, _ = load_classifier_checkpoint(ARTIFACTS_DIR / "noun_classifier.pt", device)

test_embeddings = []
test_block_keys = []
for pkl_path in sorted(TEST_EMBEDDINGS_DIR.glob("*.pkl")):
    video_id = pkl_path.stem
    with pkl_path.open("rb") as f:
        payload = pickle.load(f)
    embeddings = payload["embeddings"] if isinstance(payload, dict) else payload
    for block_idx, embedding in enumerate(embeddings):
        test_embeddings.append(embedding)
        test_block_keys.append((video_id, block_idx))

if not test_embeddings:
    raise FileNotFoundError(f"No test embedding .pkl files found in {TEST_EMBEDDINGS_DIR}")

test_X = torch.tensor(np.stack(test_embeddings), dtype=torch.float32)
with torch.no_grad():
    verb_pred_ids = verb_model(test_X.to(device)).argmax(dim=-1).cpu().numpy()
    noun_pred_ids = noun_model(test_X.to(device)).argmax(dim=-1).cpu().numpy()

predicted_verb_classes = loaded_verb_encoder.inverse_transform(verb_pred_ids)
predicted_noun_classes = loaded_noun_encoder.inverse_transform(noun_pred_ids)

prediction_rows = []
for (video_id, block_idx), verb_class, noun_class in zip(test_block_keys, predicted_verb_classes, predicted_noun_classes):
    start_frame = int(block_idx) * BLOCK_SIZE
    stop_frame = ((int(block_idx) + 1) * BLOCK_SIZE) - 1
    verb_name = verb_class_to_name.get(int(verb_class), str(verb_class))
    noun_name = noun_class_to_name.get(int(noun_class), str(noun_class))
    prediction_rows.append((video_id, start_frame, stop_frame, verb_name, noun_name))

prediction_rows.sort(key=lambda row: (row[0], row[1], row[2]))
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
with PREDICTIONS_PATH.open("w") as f:
    f.write("video_id,start_frame,stop_frame,verb,noun\n")
    for video_id, start_frame, stop_frame, verb_name, noun_name in prediction_rows:
        f.write(f"{video_id},{start_frame},{stop_frame},{verb_name},{noun_name}\n")

print(f"Wrote {len(prediction_rows)} predictions to {PREDICTIONS_PATH}")